# SIH26027 RailPlan — Complete ML + Block Optimization Notebook

This single notebook contains the complete prototype pipeline:
1. Data creation using domain-constrained synthetic railway maintenance data
2. Data inspection and EDA
3. Feature engineering and preprocessing
4. Priority prediction model
5. Maintenance risk classification model
6. Work-duration prediction model
7. Model evaluation and plots
8. Prediction for a new maintenance request
9. OR-Tools block optimization
10. Export of the optimized block plan

**Important:** the prototype uses synthetic data because operational TMS/SMMS/TDMS/COA data is not assumed to be publicly available or authorized for this project. The same logical feature pipeline can later be retrained on authorized historical exports.

## 1. Install dependencies
Run this once in a fresh environment.

In [ ]:
!pip install -q pandas numpy scikit-learn joblib matplotlib seaborn ortools

## 2. Imports and configuration

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

from ortools.sat.python import cp_model

SEED = 26027
N = 30000
rng = np.random.default_rng(SEED)

BASE = Path.cwd()
DATA_DIR = BASE / 'data'
MODEL_DIR = BASE / 'models'
REPORT_DIR = BASE / 'reports'
DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

print('Working directory:', BASE)

## 3. Data strategy

For the SIH prototype, create realistic synthetic records instead of claiming access to confidential railway data.

The synthetic data deliberately contains relationships such as:
- higher criticality → higher risk/priority
- more overdue days → higher urgency
- higher failure probability → higher risk
- higher train density → higher operational impact
- higher asset impact → higher priority
- higher complexity/resources → longer work duration

In [ ]:

departments = np.array(["Engineering", "TRD", "S&T"])

asset_types = {
    "Engineering": ["Rail", "Turnout", "TrackGeometry", "Bridge", "Drainage"],
    "TRD": ["OHE", "Insulator", "Mast", "TractionEquipment", "Feeder"],
    "S&T": ["Signal", "PointMachine", "Cable", "Interlocking", "Telecom"],
}

work_types = {
    "Engineering": ["RailRenewal", "DefectInspection", "Tamping", "GeometryCorrection", "Drainage"],
    "TRD": ["OHEInspection", "InsulatorReplacement", "MastInspection", "EquipmentMaintenance", "PowerSupply"],
    "S&T": ["SignalInspection", "CableInspection", "PointMachineMaintenance", "InterlockingTest", "TelecomMaintenance"],
}

routes = ["PUNE-LNL", "LNL-KARJAT", "KYN-PNVL", "PUNE-MMR", "DD-PUNE", "PUNE-SOL"]
seasons = ["Winter", "Summer", "Monsoon", "PostMonsoon"]

dept = rng.choice(departments, N)
asset = np.array([rng.choice(asset_types[d]) for d in dept])
work = np.array([rng.choice(work_types[d]) for d in dept])

criticality = rng.integers(1, 11, N)
days_overdue = np.maximum(0, rng.normal(4, 7, N)).round().astype(int)

failure_probability = np.clip(
    0.08
    + 0.07 * criticality
    + 0.025 * days_overdue
    + rng.normal(0, 0.10, N),
    0.01, 0.99
)

asset_impact = np.clip(
    0.25 * criticality + rng.normal(0, 1.2, N),
    1, 10
).round(1)

train_density = np.clip(rng.normal(65, 20, N), 10, 120).round(1)
passenger_trains = np.maximum(1, rng.poisson(28, N))
goods_trains = np.maximum(0, rng.poisson(10, N))

historical_failures = np.maximum(
    0,
    rng.poisson(0.25 * criticality + 0.10 * days_overdue, N)
)

last_maintenance_days = np.maximum(
    1,
    rng.normal(180 + 15 * criticality, 70, N)
).round().astype(int)

route_importance = rng.integers(1, 6, N)
dependency_count = rng.integers(0, 5, N)
crew_size = rng.integers(3, 13, N)

complexity = np.clip(
    0.45 * criticality
    + 0.25 * dependency_count
    + rng.normal(2, 1.2, N),
    1, 10
).round(1)

weather_risk = rng.choice([0, 1, 2, 3], N, p=[0.45, 0.20, 0.20, 0.15])
season = rng.choice(seasons, N)
route = rng.choice(routes, N)

equipment_required = rng.integers(1, 6, N)
location_km = rng.uniform(0, 250, N)

corridor_capacity_hours = rng.choice([2, 3, 4, 5, 6], N)

train_conflict_count = np.maximum(
    0,
    rng.poisson(np.clip(train_density / 30, 1, 8))
)

preferred_window_hour = rng.choice(
    [0, 1, 2, 3, 4, 5, 22, 23],
    N
)

print("Base features generated:", N)


## 4. Create realistic synthetic targets

In [ ]:

overdue_score = np.clip(days_overdue / 30, 0, 1)
failure_score = failure_probability
impact_score = asset_impact / 10
traffic_score = np.clip(train_density / 120, 0, 1)
history_score = np.clip(historical_failures / 10, 0, 1)
criticality_score = criticality / 10
route_score = route_importance / 5

# Synthetic domain-informed priority label.
priority_score = (
    0.25 * criticality_score
    + 0.20 * failure_score
    + 0.20 * impact_score
    + 0.15 * overdue_score
    + 0.08 * traffic_score
    + 0.07 * history_score
    + 0.05 * route_score
    + rng.normal(0, 0.025, N)
)

priority_score = np.clip(priority_score, 0, 1).round(4)

# Synthetic risk label.
risk_score = (
    0.40 * failure_probability
    + 0.20 * criticality_score
    + 0.15 * overdue_score
    + 0.10 * history_score
    + 0.10 * impact_score
    + 0.05 * traffic_score
    + rng.normal(0, 0.025, N)
)

risk_score = np.clip(risk_score, 0, 1)

priority_class = pd.cut(
    priority_score,
    bins=[-np.inf, 0.35, 0.60, 0.80, np.inf],
    labels=["Low", "Medium", "High", "Critical"]
).astype(str)

risk_class = pd.cut(
    risk_score,
    bins=[-np.inf, 0.35, 0.60, 0.80, np.inf],
    labels=["Low", "Medium", "High", "Critical"]
).astype(str)

# Synthetic actual duration.
base_duration = np.select(
    [dept == "Engineering", dept == "TRD", dept == "S&T"],
    [2.8, 2.2, 1.8],
    default=2.0
)

duration_hours = np.clip(
    base_duration
    + 0.18 * complexity
    + 0.08 * equipment_required
    + 0.03 * crew_size
    + 0.10 * weather_risk
    + rng.normal(0, 0.45, N),
    0.5, 10
).round(2)

block_feasibility = np.clip(
    1
    - 0.08 * train_conflict_count
    - 0.04 * dependency_count
    + 0.05 * (corridor_capacity_hours >= duration_hours)
    + rng.normal(0, 0.03, N),
    0, 1
).round(3)

df = pd.DataFrame({
    "work_id": [f"WR-{100000+i}" for i in range(N)],
    "department": dept,
    "asset_type": asset,
    "work_type": work,
    "route": route,
    "season": season,
    "criticality": criticality,
    "days_overdue": days_overdue,
    "failure_probability": failure_probability.round(4),
    "asset_availability_impact": asset_impact,
    "train_density": train_density,
    "passenger_trains": passenger_trains,
    "goods_trains": goods_trains,
    "historical_failures": historical_failures,
    "last_maintenance_days": last_maintenance_days,
    "route_importance": route_importance,
    "dependency_count": dependency_count,
    "crew_size": crew_size,
    "complexity": complexity,
    "weather_risk": weather_risk,
    "equipment_required": equipment_required,
    "location_km": location_km.round(2),
    "corridor_capacity_hours": corridor_capacity_hours,
    "train_conflict_count": train_conflict_count,
    "preferred_window_hour": preferred_window_hour,
    "block_feasibility": block_feasibility,
    "duration_hours_actual": duration_hours,
    "priority_score_target": priority_score,
    "priority_class_target": priority_class,
    "risk_score_target": risk_score.round(4),
    "risk_class_target": risk_class,
})

df.head()


## 5. Save the training dataset

In [ ]:
csv_path = DATA_DIR / 'maintenance_training.csv'
df.to_csv(csv_path, index=False)
print(f'Saved {len(df):,} records to {csv_path}')
print(df.shape)

## 6. Basic EDA

In [ ]:
display(df.describe(include='all').T)

fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=df, x='department', ax=ax)
ax.set_title('Maintenance work by department')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df['priority_score_target'], bins=30, kde=True, ax=ax)
ax.set_title('Synthetic priority-score distribution')
plt.tight_layout()
plt.show()

## 7. Define features and split the data

In [ ]:

numeric_features = [
    "criticality", "days_overdue", "failure_probability",
    "asset_availability_impact", "train_density", "passenger_trains",
    "goods_trains", "historical_failures", "last_maintenance_days",
    "route_importance", "dependency_count", "crew_size", "complexity",
    "weather_risk", "equipment_required", "location_km",
    "corridor_capacity_hours", "train_conflict_count",
    "preferred_window_hour", "block_feasibility"
]

categorical_features = [
    "department", "asset_type", "work_type", "route", "season"
]

feature_columns = numeric_features + categorical_features

X = df[feature_columns]

X_train, X_test, y_priority_train, y_priority_test = train_test_split(
    X, df["priority_score_target"],
    test_size=0.20,
    random_state=SEED
)

_, _, y_risk_train, y_risk_test = train_test_split(
    X, df["risk_class_target"],
    test_size=0.20,
    random_state=SEED
)

_, _, y_duration_train, y_duration_test = train_test_split(
    X, df["duration_hours_actual"],
    test_size=0.20,
    random_state=SEED
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)


## 8. Preprocessing

In [ ]:

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features
        ),
    ]
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print("Encoded train shape:", X_train_encoded.shape)


## 9. Train Model A — Maintenance Priority

In [ ]:

priority_model = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.06,
    max_leaf_nodes=31,
    l2_regularization=0.5,
    random_state=SEED
)

priority_model.fit(X_train_encoded, y_priority_train)

priority_pred = np.clip(
    priority_model.predict(X_test_encoded),
    0, 1
)

priority_metrics = {
    "MAE": mean_absolute_error(y_priority_test, priority_pred),
    "RMSE": np.sqrt(mean_squared_error(y_priority_test, priority_pred)),
    "R2": r2_score(y_priority_test, priority_pred)
}

priority_metrics


## 10. Evaluate Priority Model

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_priority_test, priority_pred, s=7, alpha=0.25)
plt.xlabel('Target priority score')
plt.ylabel('Predicted priority score')
plt.title('Priority model — predicted vs target')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'priority_predicted_vs_target.png', dpi=160)
plt.show()

## 11. Train Model B — Maintenance Risk

In [ ]:

risk_model = RandomForestClassifier(
    n_estimators=250,
    max_depth=14,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1
)

risk_model.fit(X_train_encoded, y_risk_train)

risk_pred = risk_model.predict(X_test_encoded)

risk_metrics = {
    "accuracy": accuracy_score(y_risk_test, risk_pred),
    "macro_f1": f1_score(y_risk_test, risk_pred, average="macro")
}

risk_metrics


In [ ]:
labels = ['Low', 'Medium', 'High', 'Critical']
cm = confusion_matrix(y_risk_test, risk_pred, labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot()
plt.title('Risk model confusion matrix')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'risk_confusion_matrix.png', dpi=160)
plt.show()

## 12. Train Model C — Work Duration

In [ ]:

duration_model = RandomForestRegressor(
    n_estimators=250,
    max_depth=18,
    min_samples_leaf=3,
    random_state=SEED,
    n_jobs=-1
)

duration_model.fit(X_train_encoded, y_duration_train)

duration_pred = np.clip(
    duration_model.predict(X_test_encoded),
    0.1, None
)

duration_metrics = {
    "MAE_hours": mean_absolute_error(y_duration_test, duration_pred),
    "RMSE_hours": np.sqrt(mean_squared_error(y_duration_test, duration_pred)),
    "R2": r2_score(y_duration_test, duration_pred)
}

duration_metrics


## 13. Compare model performance

In [ ]:
metrics = {
    'priority': {k: float(v) for k, v in priority_metrics.items()},
    'risk': {k: float(v) for k, v in risk_metrics.items()},
    'duration': {k: float(v) for k, v in duration_metrics.items()},
}
print(json.dumps(metrics, indent=2))
with open(REPORT_DIR / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

## 14. Save trained models

In [ ]:
joblib.dump(preprocessor, MODEL_DIR / 'preprocessor.joblib')
joblib.dump(priority_model, MODEL_DIR / 'priority_model.joblib')
joblib.dump(risk_model, MODEL_DIR / 'risk_model.joblib')
joblib.dump(duration_model, MODEL_DIR / 'duration_model.joblib')
print('Saved all models to:', MODEL_DIR)

## 15. Predict a new railway maintenance request

In [ ]:

new_work = pd.DataFrame([{
    "department": "Engineering",
    "asset_type": "Rail",
    "work_type": "RailRenewal",
    "route": "PUNE-LNL",
    "season": "Monsoon",
    "criticality": 10,
    "days_overdue": 12,
    "failure_probability": 0.91,
    "asset_availability_impact": 9.7,
    "train_density": 98,
    "passenger_trains": 40,
    "goods_trains": 13,
    "historical_failures": 5,
    "last_maintenance_days": 420,
    "route_importance": 5,
    "dependency_count": 1,
    "crew_size": 8,
    "complexity": 8.2,
    "weather_risk": 2,
    "equipment_required": 3,
    "location_km": 72.4,
    "corridor_capacity_hours": 5,
    "train_conflict_count": 2,
    "preferred_window_hour": 2,
    "block_feasibility": 0.86,
}])

new_encoded = preprocessor.transform(new_work[feature_columns])

predicted_priority = float(
    np.clip(priority_model.predict(new_encoded)[0], 0, 1)
)
predicted_risk = risk_model.predict(new_encoded)[0]
predicted_duration = float(
    max(0.1, duration_model.predict(new_encoded)[0])
)

print("=== RailPlan AI Prediction ===")
print(f"Priority score      : {predicted_priority:.3f}")
print(f"Risk class          : {predicted_risk}")
print(f"Estimated duration  : {predicted_duration:.2f} hours")


## 16. Block optimization

The optimizer is **not trained**. It uses ML predictions plus hard/soft planning constraints.

Objective concept:
- maximize high-priority maintenance value
- maximize asset-availability impact
- prefer feasible work
- penalize train conflicts
- keep total work within the available possession window

A production version should add explicit railway safety constraints, crew/resource compatibility, electrical isolation, signalling disconnection, route protection, possession rules and actual COA train paths.

In [ ]:

# Select a small planning batch for a fast demonstration.
planning_df = df.sample(18, random_state=SEED).copy()

planning_encoded = preprocessor.transform(
    planning_df[feature_columns]
)

planning_df["priority_pred"] = np.clip(
    priority_model.predict(planning_encoded),
    0, 1
)

planning_df["duration_pred"] = np.clip(
    duration_model.predict(planning_encoded),
    0.5, 8.0
)

# Demonstration corridor possession:
# 01:00–06:00 = 5 hours.
CAPACITY_HOURS = 5.0
SCALE = 10  # 0.1-hour units
CAPACITY_UNITS = int(CAPACITY_HOURS * 60 * SCALE)

model = cp_model.CpModel()

select = [
    model.NewBoolVar(f"select_{i}")
    for i in range(len(planning_df))
]

duration_units = [
    int(round(planning_df.iloc[i]["duration_pred"] * 60 * SCALE))
    for i in range(len(planning_df))
]

# Hard capacity constraint.
model.Add(
    sum(duration_units[i] * select[i] for i in range(len(planning_df)))
    <= CAPACITY_UNITS
)

# Basic feasibility gate.
for i in range(len(planning_df)):
    if planning_df.iloc[i]["block_feasibility"] < 0.35:
        model.Add(select[i] == 0)

# Objective.
objective_terms = []

for i in range(len(planning_df)):
    row = planning_df.iloc[i]

    value = int(
        10000 * row["priority_pred"]
        + 5000 * row["asset_availability_impact"] / 10
        + 2500 * row["block_feasibility"]
        - 1000 * row["train_conflict_count"]
    )

    objective_terms.append(value * select[i])

model.Maximize(sum(objective_terms))

solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 10

status = solver.Solve(model)

if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    raise RuntimeError("No feasible block plan found.")

planning_df["selected"] = [
    solver.Value(v)
    for v in select
]

optimized_plan = planning_df[
    planning_df["selected"] == 1
].copy()

optimized_plan["block_start"] = "01:00"
optimized_plan["block_end"] = "06:00"

optimized_plan[
    [
        "work_id",
        "department",
        "route",
        "work_type",
        "priority_pred",
        "duration_pred",
        "block_feasibility",
        "train_conflict_count"
    ]
]


## 17. Export optimized plan

In [ ]:
optimized_path = REPORT_DIR / 'optimized_block_plan.csv'
optimized_plan.to_csv(optimized_path, index=False)
print('Optimized plan saved to:', optimized_path)
print('Selected work items:', len(optimized_plan))

## 18. How to replace synthetic data with authorized railway data

When authorized datasets become available, replace the synthetic dataframe with governed exports having equivalent logical fields.

**Engineering / TMS:** work ID, asset ID, defect type, severity, location, chainage, overdue days, maintenance history, duration.

**TRD / TDMS:** asset/equipment ID, equipment type, defect, severity, section, isolation requirement, overdue days, duration.

**S&T / SMMS:** asset ID, signal/equipment type, defect, interlocking area, disconnection requirement, overdue days, duration.

**COA / operations:** corridor, date, available window, passenger trains, goods-train forecast, existing blocks, operational restrictions.

Do not invent live railway APIs or claim access to operational systems. Use approved exports/APIs and map them into the feature schema before retraining.

## 19. Final architecture

```text
TMS + TDMS + SMMS + COA
          ↓
   Data integration
          ↓
 Feature engineering
          ↓
 ┌────────┼────────┐
 ↓        ↓        ↓
Priority  Risk   Duration
   ML      ML       ML
 └────────┼────────┘
          ↓
   OR-Tools optimizer
          ↓
 Conflict + safety rules
          ↓
    Human approval
          ↓
    Published block
          ↓
     Field execution
          ↓
 Actual completion data
          ↓
      Retraining
```

The important distinction is: **ML predicts; optimization schedules; safety rules constrain; humans approve.**